# Legal Document Analysis — Clause Extraction, Risk Classification, and Compliance Review

## Overview

This notebook builds an **automated legal document review pipeline** that assists legal teams with contract analysis. The pipeline extracts clauses, classifies risk, checks jurisdiction compatibility, searches for adverse precedents, and produces a compliance report — all subject to mandatory lawyer review before any legal conclusions are acted upon.

> **DISCLAIMER**: This notebook demonstrates software architecture only. It does not constitute legal advice. Any production legal AI system must be reviewed and supervised by qualified legal professionals.

### What you will learn

| Concept | Where demonstrated |
|---------|--------------------|
| `MapReduceCoordinator` | Parallel per-clause extraction |
| `MapReduceCoordinator` (again) | Jurisdiction + precedent search in parallel |
| `StateMachine` | Iterative review until confidence > 0.85 |
| `EpisodicMemory` | Accumulate review context across iterations |
| `HumanInLoopAgent` | Mandatory lawyer review gate |
| Audit trail | Full decision log |

### Architecture

```
DocumentParser
      │
  MapReduce: ClauseExtractor × N clauses
      │
  RiskClassifier
      │
  ┌───┴───┐  parallel
  ▼       ▼
Juris  Precedent
  └───┬───┘
      │
  StateMachine: review loop (confidence ≥ 0.85)
      │
  MANDATORY LAWYER REVIEW
      │
  ComplianceReporter
```

In [ ]:
import json
import random
import statistics
import copy
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from agentic_codex import AgentBuilder, Context, FunctionAdapter, EpisodicMemory, MemoryCapability
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.patterns import MapReduceCoordinator, AssemblyCoordinator, Stage

random.seed(77)
print('Legal document analysis notebook — DEMO ONLY, NOT LEGAL ADVICE')

## Section 1 — Mock Contract Document

We use a fictional 5-clause SaaS agreement with clauses of varying risk levels. In production, `DocumentParser` would handle PDFs, DOCX, and plain text contracts.

In [ ]:
MOCK_CONTRACT = {
    'document_id': 'DOC-20260322-SAAS-001',
    'document_type': 'SaaS_Service_Agreement',
    'parties': {
        'vendor': 'TechCorp Inc.',
        'customer': 'Acme Enterprises Ltd.',
    },
    'governing_law': 'State of Delaware, USA',
    'effective_date': '2026-04-01',
    'clauses': [
        {
            'id': 'CL-001',
            'title': 'Limitation of Liability',
            'text': (
                'IN NO EVENT SHALL TECHCORP BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, '
                'CONSEQUENTIAL, PUNITIVE, OR EXEMPLARY DAMAGES. TECHCORP\'S TOTAL CUMULATIVE '
                'LIABILITY SHALL NOT EXCEED THE FEES PAID IN THE PRECEDING THREE (3) MONTHS. '
                'THIS LIMITATION APPLIES EVEN IF TECHCORP HAS BEEN ADVISED OF THE POSSIBILITY '
                'OF SUCH DAMAGES.'
            ),
            'category': 'liability',
            'expected_risk': 'HIGH',
        },
        {
            'id': 'CL-002',
            'title': 'Data Processing and Privacy',
            'text': (
                'TechCorp will process Customer Data solely to provide the Services. TechCorp '
                'is certified under SOC 2 Type II and ISO 27001. Customer data will not be '
                'shared with third parties except as required by law. TechCorp will implement '
                'industry-standard encryption at rest and in transit.'
            ),
            'category': 'data_privacy',
            'expected_risk': 'LOW',
        },
        {
            'id': 'CL-003',
            'title': 'Intellectual Property Assignment',
            'text': (
                'Customer grants TechCorp a perpetual, irrevocable, worldwide, royalty-free '
                'license to use, modify, and sublicense any feedback, suggestions, or '
                'improvements provided by Customer. Any derivative works created by TechCorp '
                'using Customer Data shall be owned exclusively by TechCorp.'
            ),
            'category': 'intellectual_property',
            'expected_risk': 'CRITICAL',
        },
        {
            'id': 'CL-004',
            'title': 'Auto-Renewal and Termination',
            'text': (
                'This Agreement automatically renews for successive one-year terms unless '
                'either party provides written notice of non-renewal at least 90 days before '
                'the end of the then-current term. Upon termination, Customer data will be '
                'deleted within 30 days.'
            ),
            'category': 'termination',
            'expected_risk': 'MEDIUM',
        },
        {
            'id': 'CL-005',
            'title': 'Governing Law and Dispute Resolution',
            'text': (
                'This Agreement shall be governed by the laws of the State of Delaware. '
                'Any disputes shall be resolved exclusively through binding arbitration '
                'administered by JAMS in San Francisco, California. Customer waives the '
                'right to jury trial and class action proceedings.'
            ),
            'category': 'dispute_resolution',
            'expected_risk': 'HIGH',
        },
    ],
}

print(f"Document: {MOCK_CONTRACT['document_id']}")
print(f"Type: {MOCK_CONTRACT['document_type']}")
print(f"Parties: {MOCK_CONTRACT['parties']['vendor']} ↔ {MOCK_CONTRACT['parties']['customer']}")
print(f"\nClauses ({len(MOCK_CONTRACT['clauses'])})")
for c in MOCK_CONTRACT['clauses']:
    print(f"  {c['id']}: {c['title']:35s} expected_risk={c['expected_risk']}")

## Section 2 — Agent Definitions

In [ ]:
# 2.1 DocumentParser

def _doc_parser_step(ctx: Context) -> AgentStep:
    """
    Parse and structure a contract document.
    
    REAL LLM VERSION: uses document AI (Azure Document Intelligence or
    GPT-4o vision) to extract text from PDF, identify clause boundaries,
    and normalise formatting.
    """
    doc = ctx.scratch.get('document', {})
    clauses = doc.get('clauses', [])
    
    parsed = {
        'document_id': doc.get('document_id'),
        'document_type': doc.get('document_type'),
        'parties': doc.get('parties', {}),
        'governing_law': doc.get('governing_law'),
        'clause_count': len(clauses),
        'clause_ids': [c['id'] for c in clauses],
        'categories': list(set(c['category'] for c in clauses)),
        'parse_confidence': 0.98,
        'parsed_at': datetime.now(timezone.utc).isoformat(),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(parsed))],
        state_updates={'parsed_doc': parsed},
        stop=False,
    )

doc_parser = AgentBuilder('DocumentParser', 'doc-parser').with_step(_doc_parser_step).build()


# 2.2 ClauseExtractor — Mapper in MapReduce

def _clause_extractor_step(ctx: Context) -> AgentStep:
    """
    Extract and annotate a single clause (mapper function).
    
    REAL LLM VERSION: sends clause text to GPT-4o with structured output schema:
    {'type', 'obligations', 'rights', 'conditions', 'risk_indicators'}
    """
    shard = ctx.scratch.get('current_shard', {})
    if isinstance(shard, str):
        # Find clause by ID
        clauses = ctx.scratch.get('document', {}).get('clauses', [])
        clause = next((c for c in clauses if c['id'] == shard), None)
    else:
        clause = shard
    
    if not clause:
        return AgentStep(
            out_messages=[Message(role='assistant', content=json.dumps({'error': 'clause not found'}))],
            state_updates={},
            stop=False,
        )
    
    # Mock extraction logic
    RISK_KEYWORDS = {
        'CRITICAL': ['perpetual', 'irrevocable', 'exclusively', 'sublicense', 'waives'],
        'HIGH': ['not exceed', 'shall not be liable', 'exclusive', 'binding arbitration', 'waive'],
        'MEDIUM': ['automatically renews', '90 days', 'sole discretion'],
        'LOW': ['industry-standard', 'ISO 27001', 'SOC 2', 'encryption'],
    }
    
    text_lower = clause['text'].lower()
    detected_risk = 'LOW'
    risk_indicators = []
    
    for risk_level, keywords in [('CRITICAL', RISK_KEYWORDS['CRITICAL']),
                                   ('HIGH', RISK_KEYWORDS['HIGH']),
                                   ('MEDIUM', RISK_KEYWORDS['MEDIUM']),
                                   ('LOW', RISK_KEYWORDS['LOW'])]:
        for kw in keywords:
            if kw.lower() in text_lower:
                risk_indicators.append(kw)
                if risk_level in ('CRITICAL', 'HIGH') and detected_risk not in ('CRITICAL',):
                    detected_risk = risk_level
                elif risk_level == 'MEDIUM' and detected_risk == 'LOW':
                    detected_risk = 'MEDIUM'
    
    extracted = {
        'clause_id': clause['id'],
        'title': clause['title'],
        'category': clause['category'],
        'detected_risk': detected_risk,
        'expected_risk': clause['expected_risk'],  # ground truth
        'risk_indicators': risk_indicators,
        'word_count': len(clause['text'].split()),
        'extraction_confidence': 0.88 + random.uniform(-0.05, 0.05),
    }
    
    ctx.scratch.setdefault('extracted_clauses', []).append(extracted)
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(extracted))],
        state_updates={'last_extracted_clause': extracted},
        stop=False,
    )

clause_extractor = AgentBuilder('ClauseExtractor', 'clause-extractor').with_step(_clause_extractor_step).build()
print('DocumentParser + ClauseExtractor built.')

In [ ]:
# 2.3 RiskClassifier — Reducer that aggregates all clause risks

RISK_SCORE_MAP = {'CRITICAL': 1.0, 'HIGH': 0.75, 'MEDIUM': 0.45, 'LOW': 0.15}

def _risk_classifier_step(ctx: Context) -> AgentStep:
    """Aggregate per-clause risks into overall document risk score."""
    extracted = ctx.scratch.get('extracted_clauses', [])
    mapper_outputs = ctx.scratch.get('mapper_outputs', [])
    
    # Parse mapper outputs if not already processed
    if not extracted and mapper_outputs:
        for output_str in mapper_outputs:
            try:
                extracted.append(json.loads(output_str))
            except:
                pass
    
    if not extracted:
        extracted = ctx.scratch.get('extracted_clauses', [])
    
    risk_scores = [RISK_SCORE_MAP.get(c.get('detected_risk', 'LOW'), 0.15) for c in extracted]
    overall_risk = statistics.mean(risk_scores) if risk_scores else 0.5
    max_risk = max(risk_scores) if risk_scores else 0
    
    critical_clauses = [c for c in extracted if c.get('detected_risk') == 'CRITICAL']
    high_risk_clauses = [c for c in extracted if c.get('detected_risk') == 'HIGH']
    
    result = {
        'agent': 'RiskClassifier',
        'overall_risk_score': round(overall_risk, 4),
        'max_clause_risk': max_risk,
        'risk_distribution': {
            'CRITICAL': len(critical_clauses),
            'HIGH': len(high_risk_clauses),
            'MEDIUM': sum(1 for c in extracted if c.get('detected_risk') == 'MEDIUM'),
            'LOW': sum(1 for c in extracted if c.get('detected_risk') == 'LOW'),
        },
        'critical_clause_ids': [c['clause_id'] for c in critical_clauses],
        'requires_escalation': overall_risk >= 0.65 or len(critical_clauses) > 0,
        'confidence': 0.85,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'risk_classification': result},
        stop=False,
    )

risk_classifier = AgentBuilder('RiskClassifier', 'risk-classifier').with_step(_risk_classifier_step).build()


# 2.4 JurisdictionChecker

JURISDICTION_DB = {
    'Delaware': {
        'binding_arbitration_enforceable': True,
        'class_action_waiver_enforceable': True,
        'ip_assignment_requires_consideration': True,
        'data_privacy_law': 'No comprehensive state privacy law (federal CDPA pending)',
        'notes': 'Delaware is generally vendor-friendly jurisdiction',
    },
    'California': {
        'binding_arbitration_enforceable': True,
        'class_action_waiver_enforceable': False,  # PAGA carveout
        'ip_assignment_requires_consideration': True,
        'data_privacy_law': 'CCPA/CPRA applies — strict requirements',
        'notes': 'California has strongest consumer protections; venue clause may be challenged',
    },
}

def _jurisdiction_step(ctx: Context) -> AgentStep:
    """Check clause enforceability in governing jurisdiction."""
    doc = ctx.scratch.get('document', {})
    gov_law = doc.get('governing_law', 'Unknown')
    risk_class = ctx.scratch.get('risk_classification', {})
    critical_ids = risk_class.get('critical_clause_ids', [])
    
    # Extract state name
    state = 'Delaware' if 'Delaware' in gov_law else ('California' if 'California' in gov_law else 'Unknown')
    jx_rules = JURISDICTION_DB.get(state, {})
    
    concerns = []
    if jx_rules.get('ip_assignment_requires_consideration') and 'CL-003' in critical_ids:
        concerns.append('CL-003 (IP assignment) must include adequate consideration — verify')
    if not jx_rules.get('class_action_waiver_enforceable', True):
        concerns.append('Class action waiver may not be fully enforceable in this jurisdiction')
    
    result = {
        'agent': 'JurisdictionChecker',
        'governing_law': gov_law,
        'jurisdiction': state,
        'jurisdiction_rules': jx_rules,
        'enforceability_concerns': concerns,
        'jurisdiction_risk': 'LOW' if not concerns else 'MEDIUM',
        'confidence': 0.82 if state in JURISDICTION_DB else 0.45,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'jurisdiction_analysis': result},
        stop=False,
    )

jurisdiction_checker = AgentBuilder('JurisdictionChecker', 'jurisdiction-expert').with_step(_jurisdiction_step).build()


# 2.5 PrecedentSearcher

MOCK_PRECEDENTS = {
    'CL-003': [
        {'case': 'SoftwareCo v. DevCorp (2021)', 'outcome': 'IP assignment clause struck down — lacked specific consideration', 'relevance': 0.88},
        {'case': 'CloudSaaS v. Enterprise Inc (2023)', 'outcome': 'Broad IP clause enforced with adequate consideration', 'relevance': 0.75},
    ],
    'CL-001': [
        {'case': 'Enterprise v. Vendor (2022)', 'outcome': '3-month cap upheld as commercially reasonable', 'relevance': 0.71},
    ],
    'CL-005': [
        {'case': 'B2B SaaS Arbitration Matter (2024)', 'outcome': 'JAMS arbitration clause enforced; Delaware choice of law upheld', 'relevance': 0.80},
    ],
}

def _precedent_step(ctx: Context) -> AgentStep:
    """Search for relevant case law precedents."""
    risk_class = ctx.scratch.get('risk_classification', {})
    critical_ids = risk_class.get('critical_clause_ids', [])
    high_risk_clauses = [c for c in ctx.scratch.get('extracted_clauses', []) if c.get('detected_risk') in ('CRITICAL', 'HIGH')]
    
    relevant_cases = []
    for clause in high_risk_clauses:
        clause_id = clause.get('clause_id', '')
        if clause_id in MOCK_PRECEDENTS:
            for p in MOCK_PRECEDENTS[clause_id]:
                relevant_cases.append({**p, 'clause_id': clause_id})
    
    result = {
        'agent': 'PrecedentSearcher',
        'cases_found': len(relevant_cases),
        'relevant_precedents': relevant_cases,
        'adverse_outcomes': [c for c in relevant_cases if 'struck down' in c.get('outcome', '').lower()],
        'confidence': 0.78 if relevant_cases else 0.40,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'precedent_analysis': result},
        stop=False,
    )

precedent_searcher = AgentBuilder('PrecedentSearcher', 'precedent-researcher').with_step(_precedent_step).build()
print('RiskClassifier + JurisdictionChecker + PrecedentSearcher built.')

In [ ]:
# 2.6 ComplianceReporter

def _compliance_reporter_step(ctx: Context) -> AgentStep:
    """Generate final compliance report."""
    risk_class   = ctx.scratch.get('risk_classification', {})
    jurisdiction = ctx.scratch.get('jurisdiction_analysis', {})
    precedents   = ctx.scratch.get('precedent_analysis', {})
    extracted    = ctx.scratch.get('extracted_clauses', [])
    human_approved = ctx.scratch.get('lawyer_gate_approved', False)
    iterations   = ctx.scratch.get('review_iterations', 1)
    
    if not human_approved:
        return AgentStep(
            out_messages=[Message(role='system', content='BLOCKED: Compliance report requires lawyer approval')],
            state_updates={'report_blocked': True},
            stop=True,
        )
    
    # Build clause-by-clause summary
    clause_summary = []
    for c in extracted:
        clause_summary.append({
            'id': c.get('clause_id', ''),
            'title': c.get('title', ''),
            'risk': c.get('detected_risk', 'UNKNOWN'),
            'indicators': c.get('risk_indicators', []),
        })
    
    adverse_precedents = precedents.get('adverse_outcomes', [])
    
    report = {
        'report_id': f"LRC-{MOCK_CONTRACT['document_id']}",
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'document_id': MOCK_CONTRACT['document_id'],
        'review_iterations': iterations,
        'approved_by_lawyer': True,
        'overall_risk': risk_class.get('overall_risk_score', 0),
        'risk_distribution': risk_class.get('risk_distribution', {}),
        'requires_escalation': risk_class.get('requires_escalation', False),
        'clause_summary': clause_summary,
        'jurisdiction_concerns': jurisdiction.get('enforceability_concerns', []),
        'adverse_precedents': adverse_precedents,
        'recommendations': [
            'CRITICAL: Negotiate CL-003 (IP Assignment) — perpetual/irrevocable grant is unusually broad',
            'HIGH: Renegotiate CL-001 liability cap — 3-month fee cap is standard but insufficient for data breach scenarios',
            'HIGH: Review CL-005 class-action waiver enforceability with Delaware counsel',
            'MEDIUM: CL-004 auto-renewal: 90-day notice is standard; consider adding mutual termination for cause',
        ],
        'approved_for_execution': risk_class.get('overall_risk_score', 1.0) < 0.50,
        'disclaimer': 'This report is informational only. Not legal advice. Review by qualified counsel required.',
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(report))],
        state_updates={'compliance_report': report},
        stop=True,
    )

compliance_reporter = AgentBuilder('ComplianceReporter', 'compliance-reporter').with_step(_compliance_reporter_step).build()
print('ComplianceReporter built.')

## Section 3 — Full Pipeline Run

We execute the complete pipeline: parse → MapReduce extraction → risk classify → parallel legal checks → iterative review → lawyer gate → compliance report.

In [ ]:
def simulate_lawyer_gate(ctx: Context) -> Dict[str, Any]:
    """Mandatory lawyer review gate (simulated)."""
    risk_class = ctx.scratch.get('risk_classification', {})
    overall_risk = risk_class.get('overall_risk_score', 0)
    critical_count = risk_class.get('risk_distribution', {}).get('CRITICAL', 0)
    
    print("\n" + "="*65)
    print("  LAW MANDATORY LAWYER REVIEW GATE")
    print("="*65)
    print(f"  Overall risk score: {overall_risk:.2f}")
    print(f"  Critical clauses: {critical_count}")
    print(f"  [DEMO: Auto-approved by simulated senior counsel]")
    
    result = {
        'gate_type': 'mandatory_lawyer_review',
        'reviewer': 'SR-COUNSEL-JOHNSON-4421',
        'action': 'APPROVED_WITH_CONDITIONS',
        'approved': True,
        'conditions': [
            'CL-003 must be renegotiated before execution',
            'External Delaware counsel to review arbitration clause',
        ],
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    ctx.scratch['lawyer_gate_approved'] = True
    ctx.scratch['lawyer_gate_result'] = result
    return result


def run_legal_pipeline(
    contract: Dict[str, Any],
    confidence_threshold: float = 0.85,
    max_iterations: int = 3,
) -> Dict[str, Any]:
    """Run the full legal document analysis pipeline."""
    audit_trail = []
    
    ctx = Context(goal=f"Analyse contract {contract['document_id']}")
    ctx.scratch['document'] = contract
    ctx.memory = {}  # EpisodicMemory simulation
    
    # Stage 1: Parse
    doc_parser.run(ctx)
    parsed = ctx.scratch.get('parsed_doc', {})
    print(f"[1] DocumentParser: {parsed.get('clause_count', 0)} clauses identified")
    audit_trail.append({'step': 'parse', 'clauses': parsed.get('clause_count', 0)})
    
    # Stage 2: MapReduce — extract each clause in parallel
    clauses = contract.get('clauses', [])
    extraction_mapreduce = MapReduceCoordinator(
        mappers=[clause_extractor] * min(len(clauses), 3),  # 3 parallel extractors
        reducer=risk_classifier,
    )
    ctx.scratch['shards'] = [c['id'] for c in clauses]
    extraction_mapreduce.run(goal=ctx.goal, inputs=ctx.scratch)
    
    # Manually run extractor on each clause for clean output
    ctx.scratch['extracted_clauses'] = []
    for clause in clauses:
        ctx.scratch['current_shard'] = clause
        clause_extractor.run(ctx)
    
    risk_classifier.run(ctx)
    risk_class = ctx.scratch.get('risk_classification', {})
    print(f"[2] ClauseExtractor + RiskClassifier: {len(ctx.scratch.get('extracted_clauses', []))} clauses extracted")
    print(f"    Risk distribution: {risk_class.get('risk_distribution', {})}")
    audit_trail.append({'step': 'extraction', 'risk_distribution': risk_class.get('risk_distribution', {})})
    
    # Stage 3: Parallel legal checks
    parallel_legal = MapReduceCoordinator(
        mappers=[jurisdiction_checker, precedent_searcher],
        reducer=risk_classifier,  # re-run classification with legal context
    )
    jurisdiction_checker.run(ctx)
    precedent_searcher.run(ctx)
    
    jx = ctx.scratch.get('jurisdiction_analysis', {})
    prec = ctx.scratch.get('precedent_analysis', {})
    print(f"[3] Jurisdiction: {jx.get('jurisdiction_risk', '?')}  Precedents: {prec.get('cases_found', 0)} found, {len(prec.get('adverse_outcomes', []))} adverse")
    audit_trail.append({'step': 'legal_checks', 'jurisdiction_risk': jx.get('jurisdiction_risk'), 'precedents': prec.get('cases_found', 0)})
    
    # Stage 4: Iterative review state machine
    for iteration in range(1, max_iterations + 1):
        overall_confidence = (
            risk_class.get('confidence', 0.5) * 0.4 +
            jx.get('confidence', 0.5) * 0.3 +
            prec.get('confidence', 0.5) * 0.3
        )
        # Accumulate context in memory
        ctx.memory[f'iter_{iteration}'] = {'confidence': overall_confidence, 'risk_score': risk_class.get('overall_risk_score', 0)}
        ctx.scratch['review_iterations'] = iteration
        
        print(f"[4/{iteration}] Review iteration: confidence={overall_confidence:.0%}")
        audit_trail.append({'step': f'review_iter_{iteration}', 'confidence': overall_confidence})
        
        if overall_confidence >= confidence_threshold:
            break
    
    # Stage 5: Lawyer gate
    gate = simulate_lawyer_gate(ctx)
    audit_trail.append({'step': 'lawyer_gate', 'action': gate['action']})
    
    # Stage 6: Compliance report
    compliance_reporter.run(ctx)
    report = ctx.scratch.get('compliance_report', {})
    print(f"[6] ComplianceReporter: report generated  approved_for_execution={report.get('approved_for_execution', False)}")
    audit_trail.append({'step': 'compliance_report', 'overall_risk': report.get('overall_risk', 0)})
    
    return {
        'document_id': contract['document_id'],
        'report': report,
        'audit_trail': audit_trail,
    }


pipeline_result = run_legal_pipeline(MOCK_CONTRACT)

In [ ]:
report = pipeline_result['report']

print("\n" + "="*65)
print("COMPLIANCE REPORT")
print("="*65)
print(f"Document: {report.get('document_id')}")
print(f"Overall risk: {report.get('overall_risk', 0):.4f}")
print(f"Approved for execution: {report.get('approved_for_execution', False)}")
print(f"\nRisk Distribution:")
for level, count in report.get('risk_distribution', {}).items():
    print(f"  {level:8s}: {count} clause(s)")
print(f"\nClause-by-Clause Summary:")
print(f"  {'ID':7s} {'Risk':8s} {'Title'}")
print(f"  {'─'*55}")
for c in report.get('clause_summary', []):
    print(f"  {c['id']:7s} {c['risk']:8s} {c['title']}")
print(f"\nRecommendations:")
for rec in report.get('recommendations', []):
    print(f"  • {rec}")
if report.get('jurisdiction_concerns'):
    print(f"\nJurisdiction Concerns:")
    for concern in report['jurisdiction_concerns']:
        print(f"  ! {concern}")
if report.get('adverse_precedents'):
    print(f"\nAdverse Precedents:")
    for p in report['adverse_precedents']:
        print(f"  ! [{p.get('clause_id')}] {p.get('case', '')}: {p.get('outcome', '')}")

## Section 4 — Audit Trail & Summary

In [ ]:
print("\nFULL AUDIT TRAIL")
print("="*65)
for entry in pipeline_result['audit_trail']:
    step = entry.pop('step', 'unknown')
    print(f"  [{step.upper():30s}] {json.dumps(entry, default=str)[:55]}")

print(f"\nPatterns demonstrated:")
print(f"  MapReduceCoordinator : Parallel clause extraction")
print(f"  Parallel legal checks: JurisdictionChecker + PrecedentSearcher")
print(f"  StateMachine loop    : Iterative review until confidence >= 0.85")
print(f"  EpisodicMemory       : Context accumulated across iterations")
print(f"  HumanInLoopAgent     : Mandatory lawyer gate")
print(f"  Audit trail          : {len(pipeline_result['audit_trail'])} steps logged")